# Old Faithful

Old Faithful is a geyser in Yellowstone National Park, Wyoming, USA, known for erupting at fairly regular intervals. Visitors can wait around for the next eruption, which suggests a practical question: how long until it erupts again, and does the length of the last eruption help predict that?

## The data

The data record 272 eruptions. Each row has two measurements, both in minutes:

| Column | Meaning |
|---|---|
| `eruptions` | Duration of the eruption |
| `waiting` | Waiting time until the **next** eruption |

This is the `faithful` dataset from R's `datasets` package. R's documentation cites Härdle (1991), *Smoothing Techniques with Implementation in S*, and Azzalini and Bowman (1990), "A look at some data on the Old Faithful geyser", *Applied Statistics* 39, 357–365.

A few things worth knowing before you analyse it:

- Durations such as `3.333` and `2.283` look like minutes and seconds converted to decimal minutes (3 min 20 s, 2 min 17 s), so the values are not arbitrary real numbers.
- Eruptions happen in sequence in time, so consecutive observations may not be independent. The exercises mostly treat the rows as independent; keep this in mind as a limitation.
- Take a good look at the distribution of each variable before modelling anything. What you find affects every question below.

In this package, `load_faithful_data()` returns a list of `(eruptions, waiting)` tuples. The cells below wrap them in a data frame so you can start exploring.

## How this notebook is used

The README returns to this dataset several times. Each section below follows on from the previous one: first estimate how precisely the average waiting time is known, then fit and test simple models, and finally fit a more sophisticated one and see what it explains.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats

from pyproblems.data import load_faithful_data
from pyproblems.mixture_models import (
    MultivariateGaussianMixtureParameters,
    fit_faithful_gaussian_mixture,
)
from pyproblems.statistics import (
    bootstrap_standard_error,
    fit_normal_mle,
    normal_log_likelihood,
    ols_coefficients,
    standard_error,
)

rng = np.random.default_rng(2025)

faithful = pd.DataFrame(load_faithful_data(), columns=["eruptions", "waiting"])
faithful.head(10)

,eruptions,waiting
0,3.600,79.0
1,1.800,54.0
2,3.333,74.0
3,2.283,62.0
4,4.533,85.0
5,2.883,55.0
6,4.700,88.0
7,3.600,85.0
8,1.950,51.0
9,4.350,85.0


## Standard error of the mean waiting time

Explore the data first: summarise both columns and plot a histogram of each. Then use `standard_error` to estimate the standard error of the mean waiting time, and report the mean with this uncertainty. What does the standard error tell you, and how is it different from the sample standard deviation of the waiting times? Consider whether treating the 272 eruptions as independent is reasonable.

In [ ]:
# TODO: summarise and plot the eruption durations and waiting times.
# Calculate the mean waiting time and its standard error.

## Bootstrap standard error of the mean waiting time

Draw bootstrap samples with replacement from the waiting times. Compare the standard deviation of their means with the analytic standard error from the previous section, using the seeded generator `rng` for reproducibility. Then repeat with random subsamples of the waiting times of size 10, 30 and 100, and investigate how the estimate changes with the number of resamples. 

For a fixed sample the exact bootstrap value is $\mathrm{std}(x, \mathrm{ddof}=0)/\sqrt{n}$, so separate the $\sqrt{(n-1)/n}$ difference from the analytic estimate, which does not shrink with more resamples, from Monte Carlo variation, which does.

In [ ]:
# TODO: estimate the standard error of the mean waiting time analytically and by bootstrap.
# Repeat for subsamples of size 10, 30 and 100 and several resample counts, using rng.

## A normal model for waiting times

Fit a normal distribution to the waiting times with `fit_normal_mle` and compare its estimates with the analytic maximum-likelihood estimates (the sample mean and population standard deviation). Check the optimiser's success and message. Plot a histogram of the waiting times with the fitted normal density, and report the log-likelihood. How well does the normal model describe the data? What does the histogram show that the two fitted numbers hide? We return to this when we fit a mixture model.

In [ ]:
# TODO: fit a normal distribution to the waiting times with fit_normal_mle.
# Plot the histogram with the fitted density and compare with the analytic estimates.

## Waiting time prediction

Predict the waiting time until the next eruption from the eruption duration. Fit a straight line with `ols_coefficients` (include an intercept column), interpret the intercept and slope in minutes, calculate $R^2$, and predict the wait after a 2-minute and a 4-minute eruption. Plot the data with the fitted line, and plot the residuals against the fitted values. Is a single line a good model? Explain what the plots show.

In [ ]:
# TODO: build the design matrix [1, duration] and fit waiting time with ols_coefficients.
# Report R^2 and predictions, then plot the fitted line and the residuals vs fitted values.

## Gaussian mixture

The bivariate eruption-duration and waiting-time data show structure that a one-dimensional model cannot represent, and that explains why a single regression line described them poorly. Fit the two-component bivariate Gaussian mixture with `fit_faithful_gaussian_mixture()`, starting from the rough values in the cell below (these are the ones the tests use). Plot the starting and fitted component contours with the observations, and compare the fitted log-likelihood with the reference value in the tests (about -1130.27). EM can converge to local optima, so explain what the starting values and the fitted result show.

If you're so inclined, plot after each EM iteration and animate the result to watch the convergence. It should resemble the principal example on the Wikipedia page for the EM algorithm.

As an extension, revisit the prediction: assign each eruption to a cluster using its duration alone (so the waiting time being predicted does not leak into the predictor), fit a separate line within each cluster, and compare prediction errors with the single line (for example by cross-validation).

In [ ]:
initial = MultivariateGaussianMixtureParameters(
    mean1=np.array([2.8, 75.0]),
    covariance1=np.array([[0.8, 7.0], [7.0, 70.0]]),
    mean2=np.array([3.6, 58.0]),
    covariance2=np.array([[0.8, 7.0], [7.0, 70.0]]),
    weight1=0.5,
)
# TODO: fit the bivariate Gaussian mixture with fit_faithful_gaussian_mixture(initial).
# Plot the starting and fitted contours and clusters.